In [1]:
import argparse
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix, classification_report
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC


RANDOM_STATE = 42
TARGET = "Revenue"
ROOT = Path.cwd()
MODEL_DIR = ROOT / "models"
ARTIFACT_DIR = ROOT / "artifacts"
DATA_DIR = ROOT / "data"


/opt/anaconda3/lib/python3.13/site-packages/pandas/core/computation/expressions.py:22: UserWarning: Pandas requires version '2.11.0' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import (
/opt/anaconda3/lib/python3.13/site-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [2]:
def build_preprocessor(X: pd.DataFrame) -> ColumnTransformer:
    """Build preprocessing based on the actual feature dtypes."""
    categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

    numeric_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipeline = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ])

    return ColumnTransformer(
        transformers=[
            ("numeric", numeric_pipeline, numeric_cols),
            ("categorical", categorical_pipeline, categorical_cols),
        ],
        remainder="drop",
        # Dense output is required by GaussianNB.
        sparse_threshold=0.0,
    )


In [3]:
def build_models():
    """Return the five required classifiers."""
    return {
        "Logistic Regression": LogisticRegression(
            max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
        ),
        "Decision Tree": DecisionTreeClassifier(
            max_depth=8, min_samples_split=10, min_samples_leaf=5,
            class_weight="balanced", random_state=RANDOM_STATE
        ),
        "K-Nearest Neighbors": KNeighborsClassifier(
            n_neighbors=7, weights="distance", metric="minkowski"
        ),
        "Gaussian Naive Bayes": GaussianNB(),
        "Support Vector Machine": SVC(
            kernel="rbf", C=1.0, gamma="scale", probability=True,
            class_weight="balanced", random_state=RANDOM_STATE
        ),
    }


In [4]:
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)[:, 1]

    return {
        "Accuracy": accuracy_score(y_test, predictions),
        "AUC": roc_auc_score(y_test, probabilities),
        "Precision": precision_score(y_test, predictions, zero_division=0),
        "Recall": recall_score(y_test, predictions, zero_division=0),
        "F1": f1_score(y_test, predictions, zero_division=0),
        "MCC": matthews_corrcoef(y_test, predictions),
    }, predictions, probabilities


In [5]:
def main(data_path: str):
    data_file = Path(data_path)
    if not data_file.is_absolute():
        data_file = ROOT / data_file
    if not data_file.exists():
        raise FileNotFoundError(
            f"Dataset not found: {data_file}\n"
            "Download the UCI CSV and place it at data/online_shoppers_intention.csv "
            "or pass its path with --data."
        )

    df = pd.read_csv(data_file)
    if TARGET not in df.columns:
        raise ValueError(f"Expected target column '{TARGET}'. Found: {df.columns.tolist()}")

    # Drop exact duplicate rows, if any, before splitting.
    original_rows = len(df)
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Rows loaded: {original_rows}; rows after duplicate removal: {len(df)}")

    # UCI stores Revenue as boolean. Convert it consistently to 0=no purchase, 1=purchase.
    target_values = df[TARGET]
    if target_values.dtype == bool:
        y = target_values.astype(int)
    else:
        normalized = target_values.astype(str).str.strip().str.lower()
        mapping = {
            "true": 1, "false": 0, "1": 1, "0": 0,
            "yes": 1, "no": 0
        }
        y = normalized.map(mapping)
        if y.isna().any():
            raise ValueError("Revenue must contain boolean/0-1/yes-no values only.")

    X = df.drop(columns=[TARGET]).copy()
    if X.shape[1] < 12 or len(df) < 500:
        raise ValueError(
            f"Dataset does not meet assignment minimums: {X.shape[1]} features, {len(df)} rows."
        )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
    )

    MODEL_DIR.mkdir(exist_ok=True)
    ARTIFACT_DIR.mkdir(exist_ok=True)
    DATA_DIR.mkdir(exist_ok=True)

    # Keep the test data in its original feature representation for Streamlit upload.
    test_output = X_test.copy()
    test_output[TARGET] = y_test.to_numpy()
    test_output.to_csv(ROOT / "test_data.csv", index=False)

    metrics_rows = []
    detailed_results = {}

    for display_name, classifier in build_models().items():
        print(f"\nTraining: {display_name}")
        pipeline = Pipeline(steps=[
            ("preprocessor", build_preprocessor(X_train)),
            ("classifier", classifier),
        ])
        pipeline.fit(X_train, y_train)

        metric_values, predictions, probabilities = evaluate_model(
            pipeline, X_test, y_test
        )
        metrics_rows.append({"Model": display_name, **metric_values})

        safe_name = {
            "Logistic Regression": "logistic_regression",
            "Decision Tree": "decision_tree",
            "K-Nearest Neighbors": "knn",
            "Gaussian Naive Bayes": "naive_bayes",
            "Support Vector Machine": "svm",
        }[display_name]
        joblib.dump(pipeline, MODEL_DIR / f"{safe_name}.joblib")

        detailed_results[display_name] = {
            "confusion_matrix": confusion_matrix(
                y_test, predictions, labels=[0, 1]
            ).tolist(),
            "classification_report": classification_report(
                y_test, predictions, labels=[0, 1],
                target_names=["No purchase", "Purchase"],
                output_dict=True, zero_division=0
            ),
        }
        print(pd.Series(metric_values).round(4).to_string())

    metrics_df = pd.DataFrame(metrics_rows)
    metrics_df.to_csv(ARTIFACT_DIR / "model_metrics.csv", index=False)
    with open(ARTIFACT_DIR / "detailed_results.json", "w", encoding="utf-8") as f:
        json.dump(detailed_results, f, indent=2)

    metadata = {
        "target": TARGET,
        "features": X.columns.tolist(),
        "models": {
            "Logistic Regression": "logistic_regression.joblib",
            "Decision Tree": "decision_tree.joblib",
            "K-Nearest Neighbors": "knn.joblib",
            "Gaussian Naive Bayes": "naive_bayes.joblib",
            "Support Vector Machine": "svm.joblib",
        },
        "positive_class": 1,
        "negative_class": 0,
        "random_state": RANDOM_STATE,
        "test_size": 0.20,
    }
    with open(ARTIFACT_DIR / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    print("\n=== MODEL COMPARISON (held-out test set) ===")
    print(metrics_df.round(4).to_string(index=False))
    print("\nCreated: test_data.csv, models/*.joblib, artifacts/*")


In [6]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument(
        "--data", default="/Users/mradulmehrotra/Documents/git/ml-assignment-2/data/online_shoppers_intention.csv",
        help="Path to the downloaded UCI CSV file."
    )
    args = parser.parse_args(args=[])
    main(args.data)


Rows loaded: 12330; rows after duplicate removal: 12205

Training: Logistic Regression
Accuracy     0.8513
AUC          0.9109
Precision    0.5164
Recall       0.7853
F1           0.6231
MCC          0.5536

Training: Decision Tree
Accuracy     0.8595
AUC          0.8927
Precision    0.5349
Recall       0.7827
F1           0.6355
MCC          0.5676

Training: K-Nearest Neighbors


/var/folders/7w/r_h2tkzx2r7gf0yzd5s12rmh0000gn/T/ipykernel_66028/414100125.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
/var/folders/7w/r_h2tkzx2r7gf0yzd5s12rmh0000gn/T/ipykernel_66028/414100125.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence t

Accuracy     0.8804
AUC          0.8239
Precision    0.7045
Recall       0.4058
F1           0.5150
MCC          0.4748

Training: Gaussian Naive Bayes
Accuracy     0.7108
AUC          0.8263
Precision    0.3288
Recall       0.8141
F1           0.4684
MCC          0.3772

Training: Support Vector Machine


/var/folders/7w/r_h2tkzx2r7gf0yzd5s12rmh0000gn/T/ipykernel_66028/414100125.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/dev/user_guide/migration.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
/opt/anaconda3/lib/python3.13/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Accuracy     0.8681
AUC          0.9064
Precision    0.5573
Recall       0.7644
F1           0.6446
MCC          0.5767

=== MODEL COMPARISON (held-out test set) ===
                 Model  Accuracy    AUC  Precision  Recall     F1    MCC
   Logistic Regression    0.8513 0.9109     0.5164  0.7853 0.6231 0.5536
         Decision Tree    0.8595 0.8927     0.5349  0.7827 0.6355 0.5676
   K-Nearest Neighbors    0.8804 0.8239     0.7045  0.4058 0.5150 0.4748
  Gaussian Naive Bayes    0.7108 0.8263     0.3288  0.8141 0.4684 0.3772
Support Vector Machine    0.8681 0.9064     0.5573  0.7644 0.6446 0.5767

Created: test_data.csv, models/*.joblib, artifacts/*
